# 12 · Frozen evaluation subsets: Chameleon (P4) and P1 (CPU)

**CPU runtime** (no GPU; about 0.08 compute units per hour; about 0.1 CU in all, estimated). No model runs here.

**Decisions (Dhanush, 18 Nov 2026):** compute is the constraint, so the final evaluation uses fixed subsets, frozen by the SHA-256 of their sorted file lists **before any model is scored on them**:
- **P4, Chameleon:** the authors' full `Chameleon.zip` (2.68 GB) is a Drive shortcut in `Paper 1/datasets/05_Chameleon/`.
  - The notebook copies it to the local disk, checks it (size, a CRC check of every member, about 26,000 images), unpacks it in full **to the local disk, never to Drive**, and opens every image.
  - It then freezes **2,500 real + 2,500 AI-generated** images (seed 42, salt `chameleon_v1`). If a class has fewer, the 5,000 are split in proportion to the class sizes.
  - P4 is scored on seed 42 only.
- **P1:** per GenImage generator, **1,250 real + 1,250 generated** images, from the 80% outside the CP4 development split (seed 42, salt `p1_test_v1`): 20,000 images.

**It stops and asks Dhanush** if:
- the shortcut or zip is missing;
- the zip is unreadable or holds far from 26,000 images;
- any image fails to open;
- a frozen subset already exists with a different hash.

**Results** (send to Claude):
- `Paper 1/runs/checkpoints/eval_subsets.json` (the frozen uid lists);
- `Paper 1/runs/checkpoints/eval_subsets_record.json` (counts per class, the zip check, compute).

In [ ]:
# Cell 1: mount Drive, get the code, install it
import os, subprocess, sys

DRY = os.environ.get('EVIGROUND_DRY_RUN') == '1'   # CI / local test: fake sources, temp folders
REPO_URL = 'github.com/DHANUSHKUMMARI/Paper1.git'
BRANCH = 'main'                                    # 'claude/nice-curie-3tzxzr' before the merge

if DRY:
    REPO = os.path.abspath('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
else:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    REPO = '/content/Paper1'
    if not os.path.exists(REPO):
        try:
            tok = userdata.get('GITHUB_TOKEN')       # never printed
        except Exception:
            tok = None
        url = f'https://{tok}@{REPO_URL}' if tok else f'https://{REPO_URL}'
        r = subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, url, REPO],
                           capture_output=True, text=True)
        if r.returncode != 0:                        # stderr is not shown: it can contain the URL
            raise RuntimeError('git clone failed: check GITHUB_TOKEN (Secrets panel) and BRANCH')
        subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'],
                       check=True)
    else:
        subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO + '[train]'], check=True)
    # Colab ships torchao 0.10; peft refuses a torchao older than 0.16 with an ImportError when it
    # adds the LoRA layers, although EviGround never uses torchao: remove an old one.
    import importlib, importlib.metadata as _md
    from packaging.version import Version
    try:
        if Version(_md.version('torchao')) < Version('0.16.0'):
            subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], check=True)
            importlib.invalidate_caches()
    except _md.PackageNotFoundError:
        pass

sys.path.insert(0, os.path.join(REPO, 'src'))
# Re-running in a runtime that already imported eviground would keep the OLD code in memory
# although git pulled the new one: drop it so the next import reads the files on disk.
for _name in [m for m in sys.modules if m == 'eviground' or m.startswith('eviground.')]:
    del sys.modules[_name]
print('repo:', REPO, '| branch:', BRANCH, '| dry run:', DRY)

In [ ]:
# Cell 2: settings and pre-flight
import datetime, gzip, csv, json, logging, os, shutil, time
from pathlib import Path
logging.basicConfig(level=logging.INFO, force=True)
from eviground.data import chameleon
from eviground.env import get_paths, session_info
from eviground.eval import devsplit, subsets
from eviground.utils import session

SEED = 42
CU_PER_HOUR = 0.08         # CPU runtime: read the live rate in Colab's resource panel
CU_LEFT = None             # optional: the balance shown in Colab; the pre-flight stops if EST_CU exceeds it
EST_CU = 0.1
EXPECT_IMAGES = 26000      # Chameleon: about 26,000 images (the check allows +-10%)
P4_TOTAL = subsets.P4_TOTAL
P1_PER_LABEL = subsets.P1_PER_LABEL
T0 = time.time()
info = session_info(cu_per_hour=CU_PER_HOUR)
paths = get_paths()
if DRY:   # a fake archive and a fake manifest, small numbers
    import tempfile
    from eviground.env import Paths
    _dry = Path(tempfile.gettempdir()) / 'eviground_subsets_dry'
    shutil.rmtree(_dry, ignore_errors=True)
    paths = Paths(root=_dry / 'drive' / 'Paper 1', local_data=_dry / 'data')
    EXPECT_IMAGES, P4_TOTAL, P1_PER_LABEL, EST_CU = 40, 20, 3, 0.0
paths.ensure()
CK = paths.runs / 'checkpoints'
CP1_MANIFEST = CK / 'cp1' / 'manifest_cp1.csv.gz'
DEV_SPLIT = CK / 'cp4_dev_split.json'
ZIP = paths.datasets / '05_Chameleon' / chameleon.ZIP_NAME
LOCAL = paths.local_data / 'chameleon'
SUBSETS_FILE = CK / 'eval_subsets.json'
if DRY:   # fake world: a small Chameleon-like zip, a manifest with P1 rows, a dev split
    import io, zipfile
    from PIL import Image
    ZIP.parent.mkdir(parents=True, exist_ok=True)
    buf = io.BytesIO(); Image.new('RGB', (16, 16), (1, 2, 3)).save(buf, format='PNG')
    with zipfile.ZipFile(ZIP, 'w') as zf:
        for i in range(24):
            zf.writestr(f'Chameleon/test/0_real/r{i}.png', buf.getvalue())
        for i in range(16):
            zf.writestr(f'Chameleon/test/1_fake/f{i}.png', buf.getvalue())
    rows = [{'uid': f'{g}/{lab}/{i}', 'split': 'test', 'role': 'test_P1', 'generator': g, 'label': lab}
            for g in ('adm', 'sd14') for lab in ('real', 'generated') for i in range(5)]
    CP1_MANIFEST.parent.mkdir(parents=True, exist_ok=True)
    with gzip.open(CP1_MANIFEST, 'wt', newline='') as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
    devsplit.save(devsplit.build_dev_split(rows, fraction=0.2), DEV_SPLIT)
PREFLIGHT = session.preflight(
    {'notebook': '12 evaluation subsets', 'chameleon zip': ZIP, 'unpack to (local disk)': LOCAL,
     'subsets file': SUBSETS_FILE, 'P4': f'{P4_TOTAL} images', 'P1': f'{P1_PER_LABEL} real + generated per generator'},
    paths={'Chameleon.zip (Drive shortcut)': ZIP, 'CP4 dev split': DEV_SPLIT, 'CP1 manifest': CP1_MANIFEST},
    est_cu=EST_CU, cu_left=CU_LEFT)

In [ ]:
# Cell 3: Chameleon: copy to the local disk, check, unpack in full (local), open every image
t = time.time()
LOCAL.mkdir(parents=True, exist_ok=True)
local_zip = LOCAL.parent / chameleon.ZIP_NAME
shutil.copyfile(ZIP, local_zip)   # one read through the Drive mount
COPY_S = round(time.time() - t, 1)
try:
    ZIP_REPORT = chameleon.inspect_zip(local_zip, expect_images=EXPECT_IMAGES)
except chameleon.ChameleonError as e:
    raise RuntimeError(f'Chameleon archive check failed: stop and ask Dhanush. {e}') from e
print('zip:', ZIP_REPORT)
NAMES = chameleon.unpack(local_zip, LOCAL)
UNPACK_S = round(time.time() - t - COPY_S, 1)
BAD = chameleon.unreadable(LOCAL, NAMES)
READ_S = round(time.time() - t - COPY_S - UNPACK_S, 1)
ITEMS = chameleon.items(NAMES)
UNLABELLED = len(NAMES) - len(ITEMS)
print(f'{len(NAMES)} images unpacked to {LOCAL} | unreadable {len(BAD)} | unlabelled {UNLABELLED}',
      f'| copy {COPY_S} s, unpack {UNPACK_S} s, open {READ_S} s')
if BAD or UNLABELLED:
    raise RuntimeError(f'Chameleon: {len(BAD)} unreadable, {UNLABELLED} unlabelled images '
                       f'(first: {(BAD or [n for n in NAMES if not chameleon.label_of(n)])[:5]}): stop and ask Dhanush')

In [ ]:
# Cell 4: freeze the P4 and P1 subsets (before any scoring). An existing frozen file is reused only if identical.
from eviground.data.cp1 import _read_csv_gz
P4 = subsets.p4_subset(ITEMS, total=P4_TOTAL)
MANIFEST = _read_csv_gz(CP1_MANIFEST)
SPLIT = devsplit.load(DEV_SPLIT)
P1 = subsets.p1_test_subset(MANIFEST, SPLIT['dev_uids'], per_label=P1_PER_LABEL,
                            generators=None if DRY else subsets.P1_GENERATORS)
NEW = {'P1': P1, 'P4': P4, 'frozen_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
       'dev_split_sha256': SPLIT['dev_sha256'], 'zip': ZIP_REPORT}
if SUBSETS_FILE.is_file():
    old = json.loads(SUBSETS_FILE.read_text())
    same = all(subsets.intact(old.get(k, {})) and old[k]['sha256'] == NEW[k]['sha256'] for k in ('P1', 'P4'))
    if not same:
        raise RuntimeError(f'{SUBSETS_FILE} already holds different (or altered) frozen subsets: stop and ask Dhanush')
    print('frozen subsets already exist and are identical: kept')
else:
    subsets.save(NEW, SUBSETS_FILE)
for k in ('P1', 'P4'):
    print(k, json.dumps(subsets.summary(NEW[k]))[:600])

In [ ]:
# Cell 5: the run record (counts per class, the zip check, compute)
RECORD = {
    'step': 'frozen evaluation subsets (P1, P4)', 'dry_run': DRY, 'branch': BRANCH, 'seed': SEED, 'session': info,
    'chameleon': {'zip': ZIP_REPORT, 'unpacked_images': len(NAMES), 'unreadable': len(BAD),
                  'per_class_available': P4['available'], 'per_class_frozen': P4['per_class'], 'rule': P4['rule'],
                  'copy_s': COPY_S, 'unpack_s': UNPACK_S, 'open_s': READ_S},
    'P4': subsets.summary(P4), 'P1': subsets.summary(P1),
    'utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
}
RECORD['compute'] = session.finish(CK, T0, CU_PER_HOUR, name='compute_eval_subsets.json',
                                   extra={'git_commit': info['git_commit'], 'est_cu': EST_CU})
out = CK / 'eval_subsets_record.json'
out.write_text(json.dumps(RECORD, indent=1, default=str))
print('saved', out)
print('P4 per class', P4['per_class'], '| P1', P1['images'], 'images | sha256 P4', P4['sha256'][:12], 'P1', P1['sha256'][:12])
if DRY:
    assert P4['per_class'] == {'generated': 10, 'real': 10} and P1['images'] == 12

In [ ]:
# Cell 6: release the runtime (last cell; everything above is saved on Drive)
RELEASED = session.release(DRY)
if DRY:
    assert RELEASED == 'dry', RELEASED